# 15 · Generalized CP for counts and binary data / CP generalizado para conteos y datos binarios

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/15-generalized-cp.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#b91c1c">DEEP DIVE · TAKE-HOME / ESTUDIO A FONDO · PARA DESPUÉS</span>

Every decomposition so far has minimised squared error. Deep dive 14 did too. This one asks what that choice was actually assuming:

> **Least squares is maximum likelihood under Gaussian noise of constant variance.**

Counts are not that. The variance of a count grows with its mean, two cells in five are zero, and a Gaussian model will cheerfully predict −1.6 crimes. Binary data is not that either: a 0/1 tensor fitted by squared error predicts 1.27 and −0.26.

**Generalized CP** keeps the CP model and replaces the loss, one element at a time. The factors are still what you read; the alternating structure still holds, because these losses stay convex in one factor with the others fixed. Only the question being asked of the data changes.

The data is one calendar year of Chicago crime reports, aggregated into a tensor of day-of-week × hour × community area × crime type. One of the components we fit is not a pattern in the city at all.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 .7em">Todas las descomposiciones hasta ahora han minimizado el error cuadrático. El estudio a fondo 14 también. Este pregunta qué suponía en realidad esa elección:</div><div style="margin:0 0 .7em"><b>Los mínimos cuadrados son máxima verosimilitud bajo ruido gaussiano de varianza constante.</b></div><div style="margin:0 0 .7em">Los conteos no son eso. La varianza de un conteo crece con su media, dos de cada cinco celdas son cero y un modelo gaussiano predecirá tan campante −1,6 delitos. Los datos binarios tampoco son eso: un tensor de 0 y 1 ajustado por error cuadrático predice 1,27 y −0,26.</div><div style="margin:0 0 .7em">El <b>CP generalizado</b> conserva el modelo CP y sustituye la pérdida, elemento a elemento. Los factores siguen siendo lo que se lee; la estructura alterna sigue valiendo, porque estas pérdidas siguen siendo convexas en un factor con los demás fijos. Lo único que cambia es la pregunta que se le hace a los datos.</div><div style="margin:0 0 0">Los datos son un año natural de informes de delitos de Chicago, agregados en un tensor de día de la semana × hora × área comunitaria × tipo de delito. Una de las componentes que ajustamos no es un patrón de la ciudad en absoluto.</div></div>

## What you will be able to do / Lo que podrás hacer

- State what **squared error assumes** about the data, and name two kinds of data that break it.
- Write a **generalized CP** objective as an elementwise sum of a loss `f(x, m)`.
- Write the **Poisson** and **Bernoulli** losses and their gradients, and say what each one is for.
- Explain why the **alternating structure survives** the change of loss.
- Build a real order-4 count tensor from an aggregated query, not from a downloaded file.
- Fit **GCP-Poisson** to crime counts and read the day, hour, area and type profiles.
- Recognise a component that is an artifact of **how the data was recorded**, not of the world.
- Fit **GCP-Bernoulli** to the same data binarised, and say what the two fits disagree about.
- Write the objective in **PyTorch** and let autograd replace the gradient you derived by hand.
- Explain why plain gradient descent needs a far smaller step here, and what **momentum** and a per-parameter step size buy.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0">Enunciar qué <b>supone el error cuadrático</b> sobre los datos y nombrar dos tipos de datos que lo rompen.</li><li style="margin:.35em 0">Escribir un objetivo de <b>CP generalizado</b> como suma elemento a elemento de una pérdida <code>f(x, m)</code>.</li><li style="margin:.35em 0">Escribir las pérdidas de <b>Poisson</b> y <b>Bernoulli</b> y sus gradientes, y decir para qué sirve cada una.</li><li style="margin:.35em 0">Explicar por qué <b>la estructura alterna sobrevive</b> al cambio de pérdida.</li><li style="margin:.35em 0">Construir un tensor real de orden 4 con conteos a partir de una consulta agregada, no de un fichero descargado.</li><li style="margin:.35em 0">Ajustar <b>GCP-Poisson</b> a conteos de delitos y leer los perfiles de día, hora, área y tipo.</li><li style="margin:.35em 0">Reconocer una componente que es un artefacto de <b>cómo se registraron los datos</b>, no del mundo.</li><li style="margin:.35em 0">Ajustar <b>GCP-Bernoulli</b> a los mismos datos binarizados y decir en qué discrepan los dos ajustes.</li><li style="margin:.35em 0">Escribir el objetivo en <b>PyTorch</b> y dejar que autograd sustituya al gradiente que derivaste a mano.</li><li style="margin:.35em 0">Explicar por qué el descenso por gradiente simple necesita aquí un paso mucho menor, y qué compran el <b>momento</b> y un paso por parámetro.</li></ul></div>

<!-- CORE-PATH -->
## Core path / Ruta esencial

Take-home: Exercise 1. Make the **Predict first** attempt below, run **Core prep**, then run the feedback helper before **Core activity**. Exercises 2–6 and the explorers are optional.

**You can:**

- Say what squared error assumes about the data.
- Name a prediction a Gaussian fit makes that the data cannot contain.

<details>
<summary>Español · Ruta y metas</summary>

Para después del taller: Ejercicio 1. Responde primero a **Predice primero** abajo, ejecuta **Core prep** y después el comprobador antes de **Core activity**. Los ejercicios 2–6 y los exploradores son opcionales.

**Al terminar puedes:**

- Decir qué supone el error cuadrático sobre los datos.
- Nombrar una predicción de un ajuste gaussiano que los datos no pueden contener.

</details>

## Predict first / Predice primero

**Retrieve:** Every fit so far — the pseudoinverse in section 07, the SVD in 09, Tucker in 10, CP in deep dive 14 — minimised the same thing. What was it?

Two predictions, on a street corner that averages 2 crimes an hour:

1. A model predicts 4 where the truth is 2. Squared error charges it `(2 − 4)² = 4`.
2. A different cell of the same tensor holds 2000, and the model predicts 2002. Squared error charges it `(2000 − 2002)² = 4`.

Write down whether those two misses deserve the same penalty, and why. Then: what is the smallest number a squared-error model is allowed to predict for a count?

<details>
<summary>Español · Recupera y predice</summary>

**Recupera:** Todos los ajustes hasta ahora — la pseudoinversa de la sección 07, la SVD de la 09, Tucker en la 10, CP en el estudio a fondo 14 — minimizaron lo mismo. ¿El qué?

Dos predicciones, en una esquina con una media de 2 delitos por hora:

1. Un modelo predice 4 donde la verdad es 2. El error cuadrático le cobra `(2 − 4)² = 4`.
2. Otra celda del mismo tensor vale 2000 y el modelo predice 2002. El error cuadrático le cobra `(2000 − 2002)² = 4`.

Anota si esos dos fallos merecen el mismo castigo, y por qué. Después: ¿cuál es el número más pequeño que un modelo de error cuadrático puede predecir para un conteo?

</details>

Same penalty? / ¿Mismo castigo?: ___ · Smallest prediction / Predicción mínima: ___

In [ ]:
#@title 🤔 Predict: do those two misses deserve the same penalty? / Predice: ¿merecen esos dos fallos el mismo castigo? — run me / ejecútame { display-mode: 'form' }

# --- counterexample / contraejemplo (tested in tests/test_teaching_materials.py) ---
import numpy as np

# Two misses of exactly 2, on counts three orders of magnitude apart.
pred_small = (2.0, 4.0)        # (observed, predicted)
pred_large = (2000.0, 2002.0)


def pred_squared(x, m):
    return (x - m) ** 2


def pred_deviance(x, m):
    """Poisson deviance: twice the log-likelihood gap at the same two points."""
    return 2 * (m - x + x * np.log(x / m))


assert pred_squared(*pred_small) == pred_squared(*pred_large)      # both 4.0
assert pred_deviance(*pred_small) > 600 * pred_deviance(*pred_large)
assert round(pred_deviance(*pred_small), 3) == 1.227
assert round(pred_deviance(*pred_large), 3) == 0.002
# --- end counterexample / fin del contraejemplo ---

import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import contextlib
import html as pred_html
import io

PRED_ACCENT = "#b91c1c"
PRED_SANS = "ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"
PRED_MONO = "ui-monospace,SFMono-Regular,Menlo,Consolas,monospace"


def pred_tag(text):
    """A small EN / ES marker, in words rather than in colour alone."""
    return (f'<span style="font:700 10px/1 {PRED_MONO};letter-spacing:.16em;'
            f'color:{PRED_ACCENT};opacity:.8;margin-right:9px;'
            f'vertical-align:.12em">{text}</span>')


def pred_is_measurement(line):
    """True for a printed reading, false for a sentence."""
    if ":" not in line:
        return False
    tail = line.rsplit(":", 1)[1].strip()
    return bool(tail) and (tail[0].isdigit()
                           or tail[0] in "[(-+."
                           or tail.startswith(("True", "False", "nan", "inf")))


def pred_panel(text):
    """The reveal, laid out instead of printed. Same words, given typography."""
    blocks = []
    for line in text.rstrip("\n").split("\n"):
        stripped = line.strip()
        if not stripped:
            blocks.append('<div style="height:12px"></div>')
        elif stripped.startswith(("EN:", "ES:")):
            tag, body = stripped[:2], stripped[3:].strip()
            blocks.append(
                f'<p style="margin:.55em 0;font:400 15px/1.8 {PRED_SANS}">'
                f'{pred_tag(tag)}{pred_html.escape(body)}</p>')
        elif pred_is_measurement(stripped):
            blocks.append(
                f'<div style="font:400 13.5px/2.0 {PRED_MONO};'
                f'white-space:pre-wrap">{pred_html.escape(stripped)}</div>')
        else:
            blocks.append(
                f'<p style="margin:.55em 0;font:600 15.5px/1.75 {PRED_SANS}">'
                f'{pred_html.escape(stripped)}</p>')
    return (f'<div style="border-left:4px solid {PRED_ACCENT};'
            f'background:rgba(130,130,150,.08);border-radius:0 10px 10px 0;'
            f'padding:16px 20px;margin:.4em 0 0">{"".join(blocks)}</div>')


def pred_render(choice, reveal):
    """Run the check, catch what it prints, and show it laid out."""
    caught = io.StringIO()
    with contextlib.redirect_stdout(caught):
        check_prediction(choice, reveal)
    display(widgets.HTML(pred_panel(caught.getvalue())))


pred_choice = widgets.RadioButtons(
    options=[
        ("— choose one / elige una —", None),
        ("Yes — an error of 2 is an error of 2 / Sí: un error de 2 es un error de 2", "same"),
        ("No — the small count's miss is far worse / No: el fallo en el conteo pequeño es mucho peor", "relative"),
        ("No — the large count's miss is far worse / No: el fallo en el conteo grande es mucho peor", "absolute"),
    ],
    value=None,
    description="",
    layout=widgets.Layout(width="auto", margin="0 0 6px 0"),
)

pred_reveal = widgets.Checkbox(
    value=False,
    description="Show me the answer / Muéstrame la respuesta",
    indent=False,
    layout=widgets.Layout(margin="10px 0 4px 0"),
)


def check_prediction(choice, reveal):
    if choice is None:
        print("Choose an answer first / Elige una respuesta primero.")
        return

    if not reveal:
        print("Answer saved / Respuesta guardada.")
        print("Tick the box above when you are ready / Marca la casilla de "
              "arriba cuando quieras.")
        return

    x1, m1 = pred_small
    x2, m2 = pred_large
    print("observed 2, predicted 4 / observado 2, predicho 4")
    print("  squared error / error cuadrático:", pred_squared(x1, m1))
    print("  Poisson deviance / desviación de Poisson:",
          round(pred_deviance(x1, m1), 3))
    print()
    print("observed 2000, predicted 2002 / observado 2000, predicho 2002")
    print("  squared error / error cuadrático:", pred_squared(x2, m2))
    print("  Poisson deviance / desviación de Poisson:",
          round(pred_deviance(x2, m2), 6))
    print()
    print("deviance ratio / razón de desviaciones:",
          round(pred_deviance(x1, m1) / pred_deviance(x2, m2)))
    print()
    if choice == "relative":
        print("You were right / Acertaste.")
    else:
        print("You were wrong — read on / Te equivocaste; sigue leyendo.")
    print()
    print("EN: squared error is maximum likelihood under Gaussian noise whose "
          "spread is the same everywhere. A count's spread is not: the variance "
          "of a Poisson count equals its mean, so missing by 2 on a cell that "
          "averages 2 is a different event from missing by 2 on a cell that "
          "averages 2000. The Poisson deviance charges the first miss about "
          "600 times more, and that ratio is the whole of the difference "
          "between the two models.")
    print("ES: el error cuadrático es máxima verosimilitud bajo ruido gaussiano "
          "con la misma dispersión en todas partes. La de un conteo no lo es: "
          "la varianza de un conteo de Poisson es igual a su media, así que "
          "fallar por 2 en una celda que promedia 2 es un suceso distinto de "
          "fallar por 2 en una que promedia 2000. La desviación de Poisson "
          "cobra unas 600 veces más el primer fallo, y esa razón es toda la "
          "diferencia entre los dos modelos.")

# The one <style> block in these notebooks, and the markdown rule does not
# cover it: this is *widget output*, not a markdown cell. Colab strips <style>
# from markdown -- which is why every box here is inline-styled -- but renders
# it in an output, the same path pandas' own Styler uses. Scoped to one added
# class, and if it is ever dropped the options still work.
pred_choice.add_class("pred-radio")

display(widgets.HTML(
    "<style>"
    ".pred-radio .widget-radio-box label{display:flex;align-items:flex-start;"
    "margin:0 0 13px;font:400 15px/1.6 " + PRED_SANS + "}"
    ".pred-radio input[type=radio]{flex:none;margin:4px 11px 0 0;"
    "transform:scale(1.15)}"
    "</style>"
))

pred_output = widgets.interactive_output(
    pred_render,
    {"choice": pred_choice, "reveal": pred_reveal},
)

pred_heading = widgets.HTML(
    f'<div style="font:700 11px/1.6 {PRED_MONO};letter-spacing:.18em;'
    f'color:{PRED_ACCENT};margin:2px 0 12px">'
    f'YOUR PREDICTION \u00b7 TU PREDICCI\u00d3N</div>'
)

display(widgets.VBox(
    [pred_heading, pred_choice, pred_reveal, pred_output],
    layout=widgets.Layout(padding="2px 0 14px 0"),
))

## Setup / Preparación

Three cells: imports, one query, and one squared-error fit to argue with. Nothing is downloaded as a file — the tensor is built by asking the city's data portal to count for us, which is what a count tensor **is**.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Dos celdas: importaciones y una consulta. No se descarga ningún fichero: el tensor se construye pidiéndole al portal de datos de la ciudad que cuente por nosotros, que es lo que un tensor de conteos <b>es</b>.</div>

### Core prep 1/3 · Preparación esencial

Run the next cell. / Ejecuta la siguiente celda.

In [ ]:
import time
import urllib.parse
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize

rng = np.random.default_rng(15)
EPS = 1e-10          # keeps log(m) finite when a factor is pinned at zero

print("Imports / Importaciones: OK")

### Core prep 2/3 · Preparación esencial

Run the next cell. It downloads about 3 MB. The portal does the counting, and how long that takes is up to the portal — usually a few seconds, occasionally most of a minute.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Ejecuta la siguiente celda. Descarga unos 3 MB. El portal hace el recuento, y lo que tarde depende de él: normalmente unos segundos, a veces casi un minuto.</div>

In [ ]:
# One calendar year of Chicago crime reports, counted server-side into the
# four axes we want. The window is closed, so the answer does not drift.
# Un año natural de delitos de Chicago, contados en el servidor.
CHICAGO_URL = "https://data.cityofchicago.org/resource/ijzp-q8t2.csv"

CHICAGO_QUERY = {
    "$select": ("date_extract_dow(date) as dow, "
                "date_extract_hh(date) as hour, "
                "community_area, primary_type, count(*) as n"),
    "$where": ('date >= "2023-01-01" AND date < "2024-01-01" '
               "AND community_area IS NOT NULL"),
    "$group": "dow,hour,community_area,primary_type",
    "$limit": "200000",
}
ROW_LIMIT = int(CHICAGO_QUERY["$limit"])

DAYS = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]


def fetch_crime(tries=3):
    """The aggregated table. Retries, then fails in a sentence."""
    url = CHICAGO_URL + "?" + urllib.parse.urlencode(CHICAGO_QUERY)
    for attempt in range(1, tries + 1):
        try:
            with urllib.request.urlopen(url, timeout=70) as response:
                return pd.read_csv(response)
        except Exception as error:
            if attempt == tries:
                raise RuntimeError(
                    "EN: could not reach the Chicago data portal. It rate-limits "
                    "anonymous requests, so wait a minute and run this cell "
                    "again. / ES: no se pudo acceder al portal de datos de "
                    "Chicago. Limita las peticiones anónimas: espera un minuto "
                    "y vuelve a ejecutar esta celda.") from error
            time.sleep(3 * attempt)


rows = fetch_crime()

# SODA truncates at $limit without saying so, and a truncated table builds a
# tensor that is wrong everywhere with no error to show for it. Today's query
# returns well under the limit; this is the guard for the day it does not.
if len(rows) >= ROW_LIMIT:
    raise RuntimeError(
        f"EN: the query came back at the {ROW_LIMIT}-row limit, so it was "
        "truncated and the tensor would be built from part of the year. Raise "
        "$limit and run this cell again. / ES: la consulta llegó al límite de "
        f"{ROW_LIMIT} filas, así que se truncó. Sube $limit y vuelve a "
        "ejecutar esta celda.")

# The ten commonest crime types, so the type axis is short enough to read.
TYPES = sorted(rows.groupby("primary_type")["n"].sum().nlargest(10).index)
rows = rows[rows.primary_type.isin(TYPES)]
AREAS = sorted(rows.community_area.unique())

area_at = {area: i for i, area in enumerate(AREAS)}
type_at = {name: i for i, name in enumerate(TYPES)}

T = np.zeros((7, 24, len(AREAS), len(TYPES)))
for dow, hour, area, name, n in rows[
        ["dow", "hour", "community_area", "primary_type", "n"]].itertuples(
            index=False):
    T[int(dow), int(hour), area_at[area], type_at[name]] += n

print("Tensor / Tensor:", T.shape)
print("Axes / Ejes: day-of-week, hour, community area, crime type")
print("Reports / Denuncias:", int(T.sum()))
print("Empty cells / Celdas vacías:", f"{100 * (T == 0).mean():.1f}%")

### Core prep 3/3 · Preparación esencial

Run the next cell. It is deep dive 14's alternating least squares, restated so this notebook stands alone, pointed at a tensor of counts.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Ejecuta la siguiente celda: son los mínimos cuadrados alternos del estudio a fondo 14, repetidos para que este cuaderno se sostenga solo, apuntados a un tensor de conteos.</div>

In [ ]:
# Deep dive 14's fitter, unchanged. It minimises squared error, and nothing
# about it knows these numbers are counts.
# El ajustador del estudio 14, sin cambios: minimiza el error cuadrático.


def unfold(A, mode):
    """The mode-n unfolding: that axis becomes the rows, the rest the columns."""
    return np.moveaxis(A, mode, 0).reshape(A.shape[mode], -1)


def khatri_rao(*factors):
    """Column-wise Kronecker product of any number of factor matrices."""
    out = factors[0]
    for F in factors[1:]:
        out = (out[:, None, :] * F[None, :, :]).reshape(-1, out.shape[1])
    return out


def cp_einsum(ndim):
    """The einsum string that rebuilds a tensor of this order from its factors."""
    letters = "ijkl"[:ndim]
    return ",".join(c + "r" for c in letters) + "->" + letters


def als_fit(A, rank, sweeps=40, seed=0):
    """CP by alternating least squares. Squared error, no constraints."""
    start = np.random.default_rng(seed)
    factors = [np.abs(start.standard_normal((dim, rank))) for dim in A.shape]
    spec = cp_einsum(A.ndim)
    norm_A = np.linalg.norm(A)
    errors = []
    for _ in range(sweeps):
        for mode in range(A.ndim):
            others = [factors[m] for m in range(A.ndim) if m != mode]
            M = khatri_rao(*others)
            gram = others[0].T @ others[0]
            for other in others[1:]:
                gram = gram * (other.T @ other)
            factors[mode] = unfold(A, mode) @ M @ np.linalg.pinv(gram)
        errors.append(float(np.linalg.norm(A - np.einsum(spec, *factors))
                            / norm_A))
    return factors, np.array(errors)


gauss_factors, gauss_err = als_fit(T, rank=3, sweeps=40, seed=0)
M_gauss = np.einsum(cp_einsum(T.ndim), *gauss_factors)

print("relative error / error relativo:", round(float(gauss_err[-1]), 4))
print("predicted range / rango predicho:",
      round(float(M_gauss.min()), 3), "to /  a", round(float(M_gauss.max()), 3))

## Squared error is an assumption / El error cuadrático es una hipótesis

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

Every fit in this workshop has minimised $\lVert \mathcal{T} - \mathcal{M} \rVert_F^2$. That is not what "fit" means; it is one particular answer to the question *how surprised should I be by this residual?*

Write it out as a sum over entries and the assumption is visible:

$$
\lVert \mathcal{T} - \mathcal{M} \rVert_F^2 \;=\; \sum_{i} (x_i - m_i)^2
$$

Minimising that is **maximum likelihood** under one model of the world: each entry is its model value plus Gaussian noise, and the spread of that noise is the same everywhere. Both halves of that sentence are wrong for counts.

- The spread is not constant. A Poisson count has variance equal to its mean, so a cell averaging 2 and a cell averaging 2000 disagree about what a miss of 2 means — by a factor of about 600, which the predict-first cell measured.
- The support is wrong. A Gaussian is defined on the whole line, so the model is free to predict −1.2 crimes, and on this tensor it does exactly that on more than a thousand cells, nearly all of them cells whose true count is zero.

None of that makes squared error *wrong*. It makes it a choice, and a choice you have been making by default since section 07. The rest of this notebook is about making it deliberately.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Todos los ajustes de este taller han minimizado el error cuadrático de Frobenius. Eso no es lo que significa «ajustar»: es una respuesta concreta a la pregunta <i>¿cuánto debería sorprenderme este residuo?</i>
<br><br>
Escrito como suma sobre entradas, la hipótesis se ve: minimizarlo es <b>máxima verosimilitud</b> bajo un modelo del mundo en el que cada entrada es su valor de modelo más ruido gaussiano, con la misma dispersión en todas partes. Las dos mitades de esa frase son falsas para conteos. La dispersión no es constante: la varianza de un conteo de Poisson es igual a su media. Y el soporte está mal: una gaussiana vive en toda la recta, así que el modelo puede predecir −1,2 delitos, y en este tensor lo hace en más de mil celdas, casi todas con conteo verdadero cero.
<br><br>
Nada de esto hace que el error cuadrático sea <i>incorrecto</i>. Lo convierte en una elección, y una que llevas haciendo por defecto desde la sección 07. El resto del cuaderno trata de hacerla a propósito.</div>

### Same model, different question / Mismo modelo, otra pregunta

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

The CP model on the left never changes: the same factors, the same rank-1 terms, the same prediction in every cell. What changes is the column on the right — what each loss charges for the same residual. Under squared error the two highlighted cells cost the same. Under the Poisson loss they do not.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-15-loss.gif" alt="An animation in four frames of a small grid of counts beside a model of it, two cells highlighted that both miss by 2 on counts of 2 and 40. The second frame adds the squared-error penalty, which is 4 for both. The third replaces it with the Poisson deviance, which is 1.2 for one and 0.1 for the other. The fourth shows those two penalty grids side by side." style="max-width:100%;border-radius:8px;margin:1.4em 0">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El modelo CP de la izquierda no cambia nunca: los mismos factores, los mismos términos de rango 1, la misma predicción en cada celda. Lo que cambia es la columna de la derecha: lo que cada pérdida cobra por el mismo residuo. Bajo error cuadrático las dos celdas resaltadas cuestan lo mismo. Bajo la pérdida de Poisson, no.</div>

### Zero, one, and the odds between / Cero, uno y las probabilidades entre medias

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

A tensor of ones and zeros, fitted two ways. Squared error puts the prediction on the number line and it lands outside <code>[0, 1]</code>. The Bernoulli loss models the <b>odds</b> instead, so <code>m / (1 + m)</code> is a probability by construction and no frame of the animation can leave the interval.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-15-binary.gif" alt="An animation of a binary tensor of zeros and ones. The first frames show a squared-error fit whose predicted cells are labelled with values below zero and above one. The later frames show the same cells under a Bernoulli odds model, every predicted probability inside the unit interval." style="max-width:100%;border-radius:8px;margin:1.4em 0">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Un tensor de unos y ceros, ajustado de dos maneras. El error cuadrático pone la predicción en la recta real y cae fuera de <code>[0, 1]</code>. La pérdida de Bernoulli modela en cambio las <b>probabilidades relativas</b>, así que <code>m / (1 + m)</code> es una probabilidad por construcción y ningún fotograma puede salirse del intervalo.</div>

In [ ]:
#@title ⏸️ Step through the animations / Recorre las animaciones { display-mode: 'form' }

# Plumbing, not a lesson. The two animations above loop forever and a GIF
# cannot be paused -- so this fetches the same frames and hands them over one
# at a time, at whatever pace you read at.
# Plomería, no una lección: trae los mismos fotogramas y los entrega de uno en
# uno, al ritmo al que leas.

import io
import urllib.request

import ipywidgets as widgets
from IPython.display import display
from PIL import Image

gif_urls = [
    "https://project-delphi.github.io/tensors-workshop/images/cube-15-loss.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-15-binary.gif",
]


def gif_frames(url):
    """Every frame of an animated GIF, as PNG bytes."""
    with urllib.request.urlopen(url, timeout=30) as response:
        gif = Image.open(io.BytesIO(response.read()))
    out = []
    try:
        while True:
            buffer = io.BytesIO()
            gif.convert("RGB").save(buffer, format="PNG")
            out.append(buffer.getvalue())
            gif.seek(gif.tell() + 1)
    except EOFError:
        pass
    return out


try:
    gif_cache = {url: gif_frames(url) for url in gif_urls}
except Exception as error:  # offline, or the site is down
    print("EN: could not reach the site, so there are no frames to step "
          "through.", error)
    print("ES: no se pudo acceder al sitio, así que no hay fotogramas que "
          "recorrer.", error)
else:
    gif_pick = widgets.Dropdown(
        options=[(url.rsplit("/", 1)[1], url) for url in gif_urls],
        description="Animation / Animación:",
        style={"description_width": "180px"},
    )
    gif_step = widgets.IntSlider(
        min=1, max=len(gif_cache[gif_urls[0]]), value=1,
        description="Frame / Fotograma:",
        style={"description_width": "180px"},
        continuous_update=False,
    )
    gif_prev = widgets.Button(description="◀ Prev")
    gif_next = widgets.Button(description="Next ▶")
    # An Image widget, deliberately, and never widgets.Output: a payload
    # leaving an Output widget makes nbclient wait out the whole cell timeout
    # (see scripts/test_notebooks.py). This one is a plain bytes trait.
    gif_view = widgets.Image(format="png",
                             layout=widgets.Layout(max_width="100%"))

    def gif_show(*_):
        frames = gif_cache[gif_pick.value]
        gif_step.max = len(frames)
        gif_view.value = frames[min(gif_step.value, len(frames)) - 1]

    def gif_bump(delta):
        def click(_):
            frames = gif_cache[gif_pick.value]
            gif_step.value = (gif_step.value - 1 + delta) % len(frames) + 1
        return click

    gif_prev.on_click(gif_bump(-1))
    gif_next.on_click(gif_bump(+1))
    gif_pick.observe(gif_show, names="value")
    gif_step.observe(gif_show, names="value")
    gif_show()

    display(widgets.VBox([
        gif_pick,
        widgets.HBox([gif_prev, gif_step, gif_next]),
        gif_view,
    ]))

## Exercise 1 — what does a Gaussian fit predict? / Ejercicio 1 — ¿qué predice un ajuste gaussiano?

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

Before changing anything, find out what the default costs you. `M_gauss` is already in memory: the rank-3 squared-error fit prep 3/3 just made. The question is what it predicts.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Antes de cambiar nada, averigua qué te cuesta lo de siempre. <code>M_gauss</code> ya está en memoria: el ajuste de error cuadrático de rango 3 que acaba de hacer la preparación 3/3.</div>

In [ ]:
# Feedback helper / Comprobador: run me before the Core activity.
# Ejecútame antes de la actividad esencial.


def check_gaussian(n_negative, worst, zero_negative, sq_small, sq_large,
                   dev_small, dev_large):
    """Says what is right, what is wrong, and what to look at next."""
    notes = []

    def ok(condition, good, bad):
        notes.append(("✅ " + good) if condition else ("❌ " + bad))

    # Every bar below is measured from the data the reader actually has, not
    # written down. T is a live query against a portal that reclassifies and
    # expunges historical records, so a hardcoded 1000 would eventually tell a
    # student whose answer is right that it is wrong -- and tell them to run
    # the line they just ran.
    true_negative = int((M_gauss < 0).sum())
    true_worst = float(M_gauss.min())
    true_zero_negative = int(((T == 0) & (M_gauss < 0)).sum())

    ok(n_negative == true_negative,
       f"{n_negative} predicted cells are negative. A count cannot be.",
       f"n_negative is {n_negative}; there are {true_negative}. Count the "
       "entries of M_gauss below zero with (M_gauss < 0).sum().")
    ok(abs(worst - true_worst) < 1e-9,
       f"The worst prediction is {worst:.3f} crimes.",
       f"worst should be M_gauss.min(), which is {true_worst:.3f}; you have "
       f"{worst}.")
    ok(zero_negative == true_zero_negative,
       f"{zero_negative} of those {n_negative} sit on cells whose true count "
       "is zero — the model is spending its freedom where there is nothing.",
       f"zero_negative should count cells with T == 0 and M_gauss < 0, which "
       f"is {true_zero_negative}; you have {zero_negative}.")
    ok(sq_small == sq_large,
       f"Squared error charges both misses {sq_small:g}.",
       "Both misses are exactly 2, so squared error charges (2)² for each.")
    # This one is arithmetic on two literals, so it cannot drift with the data.
    ok(dev_small > 100 * dev_large,
       f"The Poisson deviance charges the small-count miss "
       f"{dev_small / dev_large:.0f} times more.",
       "The Poisson deviance is 2 * (m - x + x * log(x / m)); it should be far "
       "larger for the miss on the small count.")

    print("\n".join(notes))
    print()
    print("EN: the negative predictions are not a bug in the optimiser. They "
          "are the model doing exactly what it was asked to do, under an "
          "assumption nobody checked.")
    print("ES: las predicciones negativas no son un fallo del optimizador. Son "
          "el modelo haciendo exactamente lo que se le pidió, bajo una "
          "hipótesis que nadie comprobó.")

### Core activity · Actividad esencial

**Predict → Run → Explain → Check**

1. **Predict.** Write down, before running anything, roughly how many of the 131,040 predicted cells you expect a rank-3 squared-error fit to place below zero, and where in the tensor you expect them.
2. **Run.** Complete Exercise 1 below. `M_gauss` is the rank-3 unconstrained squared-error fit from prep 3/3. Count the negatives, find the worst one, and count how many land on cells whose true count is zero. Then compute both costs for the two misses from the predict-first cell.
3. **Explain.** In one sentence, say what the model is assuming about the data that lets it predict a negative count at all.
4. **Check.** Call `check_gaussian(n_negative, worst, zero_negative, sq_small, sq_large, dev_small, dev_large)` and compare against your written prediction before opening the solution.

<details>
<summary>Español · Predice → Ejecuta → Explica → Comprueba</summary>

1. **Predice.** Anota, antes de ejecutar nada, cuántas de las 131.040 celdas predichas esperas que un ajuste de error cuadrático de rango 3 coloque por debajo de cero, y dónde esperas que estén.
2. **Ejecuta.** Completa el Ejercicio 1. `M_gauss` es el ajuste de error cuadrático sin restricciones, de rango 3, de la preparación 3/3. Cuenta los negativos, encuentra el peor y cuenta cuántos caen en celdas de conteo verdadero cero. Después calcula ambos costes para los dos fallos de la celda de predicción.
3. **Explica.** En una frase, di qué supone el modelo sobre los datos para poder predecir siquiera un conteo negativo.
4. **Comprueba.** Llama a `check_gaussian(n_negative, worst, zero_negative, sq_small, sq_large, dev_small, dev_large)` y compara con tu predicción escrita antes de abrir la solución.

</details>

Prediction / Predicción: ___  
Evidence / Evidencia: ___  
Revised explanation / Explicación revisada: ___

<details>
<summary>Hint 1 / Pista 1 · if stuck after an attempt / si te atascas tras intentarlo</summary>

`(M_gauss < 0).sum()` counts them; `M_gauss.min()` is the worst. For the overlap with true zeros, combine two boolean arrays with `&`.

`(M_gauss < 0).sum()` los cuenta; `M_gauss.min()` es el peor. Para el solapamiento con los ceros verdaderos, combina dos arreglos booleanos con `&`.

</details>

<details>
<summary>Hint 2 / Pista 2 · implementation / implementación</summary>

The Poisson deviance of observing `x` when the model says `m` is `2 * (m - x + x * np.log(x / m))`. It is zero when `m == x` and grows with the *relative* size of the miss, not the absolute one.

La desviación de Poisson de observar `x` cuando el modelo dice `m` es `2 * (m - x + x * np.log(x / m))`. Vale cero cuando `m == x` y crece con el tamaño *relativo* del fallo, no con el absoluto.

</details>

In [ ]:
# TODO 1 / TAREA 1
#
# M_gauss is the rank-3 unconstrained squared-error fit of T from prep 3/3.
#
# 1. n_negative   = how many entries of M_gauss are below zero.
# 2. worst        = the most negative prediction.
# 3. zero_negative = how many of those negatives sit on cells where T == 0.
#
# 4. For the two misses from the predict-first cell -- observing 2 and
#    predicting 4, observing 2000 and predicting 2002 -- compute
#       sq_small, sq_large    the squared errors
#       dev_small, dev_large  the Poisson deviances,
#                             2 * (m - x + x * np.log(x / m))
#
# Then run:
#   check_gaussian(n_negative, worst, zero_negative,
#                  sq_small, sq_large, dev_small, dev_large)

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }

n_negative = int((M_gauss < 0).sum())
worst = float(M_gauss.min())
zero_negative = int(((T == 0) & (M_gauss < 0)).sum())


def poisson_deviance(x, m):
    """Twice the log-likelihood gap between the model and a perfect fit."""
    return 2 * (m - x + x * np.log(x / m))


sq_small = (2.0 - 4.0) ** 2
sq_large = (2000.0 - 2002.0) ** 2
dev_small = poisson_deviance(2.0, 4.0)
dev_large = poisson_deviance(2000.0, 2002.0)

check_gaussian(n_negative, worst, zero_negative,
               sq_small, sq_large, dev_small, dev_large)

<details>
<summary><strong>What did Exercise 1 show? / ¿Qué mostró el Ejercicio 1?</strong></summary>

**Over a thousand cells are predicted negative, and three quarters of them are cells whose true count is zero.** That is not a numerical accident. Squared error treats a zero like any other number, so the cheapest way to fit a tensor that is 41% zeros is to let the model dip below the floor and average out.

**The two misses cost the same, and they should not.** An error of 2 on a cell that averages 2 is the model being wrong about the order of magnitude. An error of 2 on a cell holding 2000 is rounding. Squared error cannot tell them apart because it was never told the spread depends on the mean.

**Nothing here is a criticism of ALS.** The optimiser found a good minimum of the objective it was given. The objective was the problem, and it was chosen by default.

<br>

**ES.** **Más de mil celdas se predicen negativas, y tres cuartas partes son celdas con conteo verdadero cero.** No es un accidente numérico: el error cuadrático trata un cero como cualquier otro número, así que la forma más barata de ajustar un tensor con 41% de ceros es dejar que el modelo baje del suelo y compense. **Los dos fallos cuestan lo mismo, y no deberían**: un error de 2 en una celda que promedia 2 es equivocarse de orden de magnitud; un error de 2 en una celda de 2000 es redondear. **Nada de esto critica a ALS**: el optimizador encontró un buen mínimo del objetivo que se le dio. El objetivo era el problema, y se eligió por defecto.

</details>

## Generalized CP: keep the model, change the question / CP generalizado: conserva el modelo, cambia la pregunta

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

The CP model does not change. The prediction in cell $(i, j, k, l)$ is still the same sum of rank-1 terms, and the factors are still the profiles you learnt to read in deep dive 14:

$$
m_{ijkl} \;=\; \sum_{r=1}^{R} a_r[i] \, b_r[j] \, c_r[k] \, d_r[l]
$$

What changes is the one line that says how bad a prediction is. **Generalized CP** replaces the squared error with any elementwise loss $f(x, m)$ and minimises

$$
F(\mathcal{M}) \;=\; \sum_{i,j,k,l} f\bigl(x_{ijkl},\; m_{ijkl}\bigr)
$$

Three choices of $f$, each the negative log-likelihood of a different model of the data:

| data | loss $f(x, m)$ | $\partial f / \partial m$ | model |
|---|---|---|---|
| real, constant spread | $(x - m)^2$ | $-2(x - m)$ | Gaussian |
| counts | $m - x \log m$ | $1 - x/m$ | Poisson |
| binary | $\log(m + 1) - x \log m$ | $\frac{1}{m+1} - \frac{x}{m}$ | Bernoulli, odds link |

That table is Table 1 of Hong, Kolda and Duersch (2020), cut down to three rows. It has a dozen more — Gamma, Rayleigh, negative binomial, Huber, zero-truncated Poisson — and the machinery below does not care which one you hand it.

**The alternating structure survives.** With every factor but one held fixed, $m$ is *linear* in the remaining factor, and each of these losses is convex in $m$. A convex function of a linear function is convex, so the subproblem is still convex — the guarantee deep dive 14 leant on. What is lost is the closed form: there is no pseudoinverse for a Poisson loss, so each subproblem is solved numerically.

In practice you do not even have to alternate. The gradient with respect to a whole factor matrix is one Khatri–Rao product away:

$$
\frac{\partial F}{\partial A} \;=\; \left[\frac{\partial f}{\partial \mathcal{M}}\right]_{(0)} (B \odot C \odot D)
$$

which is the same unfolding and the same Khatri–Rao product ALS used — so you can hand the whole thing to a general optimiser. That is what `gcp_opt` in Kolda's own `pyttb` does, and it is what the next exercise does in thirty lines.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El modelo CP no cambia: la predicción de una celda sigue siendo la misma suma de términos de rango 1, y los factores siguen siendo los perfiles que aprendiste a leer en el estudio 14. Lo que cambia es la línea que dice cuán mala es una predicción. El <b>CP generalizado</b> sustituye el error cuadrático por cualquier pérdida elemento a elemento <code>f(x, m)</code> y minimiza su suma. La tabla de arriba da tres: gaussiana para datos reales de dispersión constante, Poisson para conteos, Bernoulli para datos binarios. Es la tabla 1 de Hong, Kolda y Duersch (2020), recortada a tres filas de la docena que trae.
<br><br>
<b>La estructura alterna sobrevive.</b> Con todos los factores fijos menos uno, <code>m</code> es <i>lineal</i> en el que queda, y cada una de estas pérdidas es convexa en <code>m</code>; una función convexa de una función lineal es convexa, así que el subproblema sigue siendo convexo. Lo que se pierde es la forma cerrada: no hay pseudoinversa para una pérdida de Poisson.
<br><br>
En la práctica ni siquiera hace falta alternar: el gradiente respecto de toda una matriz factor es un producto Khatri–Rao — el mismo desplegado y el mismo producto que usaba ALS — así que puedes darle el problema entero a un optimizador general. Eso es lo que hace <code>gcp_opt</code> en el propio <code>pyttb</code> de Kolda, y lo que hace el ejercicio siguiente en treinta líneas.</div>

## Exercise 2 — write the losses and fit with them / Ejercicio 2 — escribe las pérdidas y ajusta con ellas

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

Optional, and the cell the rest of the notebook runs on. Three losses, three gradients, one optimiser.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Opcional, y la celda sobre la que corre el resto del cuaderno. Tres pérdidas, tres gradientes, un optimizador.</div>

In [ ]:
# TODO 2 / TAREA 2
#
# 1. LOSSES = a dict mapping a name to (f, df/dm), each taking (x, m) arrays:
#       "gaussian"   (x - m) ** 2                  -2 * (x - m)
#       "poisson"    m - x * log(m + EPS)          1 - x / (m + EPS)
#       "bernoulli"  log(m + 1) - x * log(m + EPS) 1 / (m + 1) - x / (m + EPS)
#    EPS keeps the logarithm finite when a factor is pinned at zero.
#
# 2. gcp_fit(A, rank, loss, maxiter, seed, positive) -> (factors, result)
#    Pack every factor matrix into one long vector, hand scipy's L-BFGS-B the
#    objective and its gradient together, and unpack the answer.
#
#    The gradient with respect to factor k is
#        unfold(dF/dM, k) @ khatri_rao(*the other factors, in axis order)
#
#    `positive=True` bounds every entry at EPS, which is what keeps m positive
#    so the logarithms are defined. For the Gaussian fit, pass positive=False.
#
# 3. Fit rank 3 with the Poisson loss and confirm no prediction is negative.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }

LOSSES = {
    "gaussian": (lambda x, m: (x - m) ** 2,
                 lambda x, m: -2 * (x - m)),
    "poisson": (lambda x, m: m - x * np.log(m + EPS),
                lambda x, m: 1 - x / (m + EPS)),
    "bernoulli": (lambda x, m: np.log(m + 1) - x * np.log(m + EPS),
                  lambda x, m: 1 / (m + 1) - x / (m + EPS)),
}


def gcp_fit(A, rank, loss="poisson", maxiter=800, seed=0, positive=True):
    """CP under any elementwise loss, by L-BFGS-B on all factors at once."""
    value_of, slope_of = LOSSES[loss]
    spec = cp_einsum(A.ndim)
    shapes = [(dim, rank) for dim in A.shape]
    sizes = [dim * rank for dim in A.shape]

    start = np.random.default_rng(seed)
    scale = (max(A.mean(), EPS) / rank) ** (1 / A.ndim)
    packed = np.concatenate([np.abs(start.standard_normal(n)) * scale
                             for n in sizes])

    def unpack(vector):
        out, at = [], 0
        for shape, n in zip(shapes, sizes):
            out.append(vector[at:at + n].reshape(shape))
            at += n
        return out

    def objective(vector):
        factors = unpack(vector)
        model = np.einsum(spec, *factors)
        slope = slope_of(A, model)
        grads = [unfold(slope, k) @ khatri_rao(
            *[factors[j] for j in range(A.ndim) if j != k])
            for k in range(A.ndim)]
        return (value_of(A, model).sum(),
                np.concatenate([g.ravel() for g in grads]))

    result = minimize(
        objective, packed, jac=True, method="L-BFGS-B",
        bounds=[(EPS, None)] * packed.size if positive else None,
        options={"maxiter": maxiter, "maxfun": 10 * maxiter})
    return unpack(result.x), result


poisson_factors, poisson_res = gcp_fit(T, rank=3, loss="poisson", maxiter=800)
M_poisson = np.einsum(cp_einsum(T.ndim), *poisson_factors)

print("converged / convergió:", poisson_res.success,
      "  iterations / iteraciones:", poisson_res.nit)
print("smallest prediction / predicción mínima:",
      round(float(M_poisson.min()), 6))
print("negatives / negativos:", int((M_poisson < 0).sum()))

In [ ]:
# Visible on purpose: everything below uses these, and a reader who never
# opened the solution still has to be able to run it.
# Visible a propósito: todo lo de abajo lo usa.

LOSSES = {
    "gaussian": (lambda x, m: (x - m) ** 2,
                 lambda x, m: -2 * (x - m)),
    "poisson": (lambda x, m: m - x * np.log(m + EPS),
                lambda x, m: 1 - x / (m + EPS)),
    "bernoulli": (lambda x, m: np.log(m + 1) - x * np.log(m + EPS),
                  lambda x, m: 1 / (m + 1) - x / (m + EPS)),
}


def gcp_fit(A, rank, loss="poisson", maxiter=800, seed=0, positive=True):
    """CP under any elementwise loss, by L-BFGS-B on all factors at once."""
    value_of, slope_of = LOSSES[loss]
    spec = cp_einsum(A.ndim)
    shapes = [(dim, rank) for dim in A.shape]
    sizes = [dim * rank for dim in A.shape]

    start = np.random.default_rng(seed)
    scale = (max(A.mean(), EPS) / rank) ** (1 / A.ndim)
    packed = np.concatenate([np.abs(start.standard_normal(n)) * scale
                             for n in sizes])

    def unpack(vector):
        out, at = [], 0
        for shape, n in zip(shapes, sizes):
            out.append(vector[at:at + n].reshape(shape))
            at += n
        return out

    def objective(vector):
        factors = unpack(vector)
        model = np.einsum(spec, *factors)
        slope = slope_of(A, model)
        grads = [unfold(slope, k) @ khatri_rao(
            *[factors[j] for j in range(A.ndim) if j != k])
            for k in range(A.ndim)]
        return (value_of(A, model).sum(),
                np.concatenate([g.ravel() for g in grads]))

    result = minimize(
        objective, packed, jac=True, method="L-BFGS-B",
        bounds=[(EPS, None)] * packed.size if positive else None,
        options={"maxiter": maxiter, "maxfun": 10 * maxiter})
    return unpack(result.x), result


def normalise(factors):
    """Unit-norm profiles plus one weight each, the way deep dive 14 insists."""
    scales = [np.linalg.norm(F, axis=0) for F in factors]
    profiles = [F / s for F, s in zip(factors, scales)]
    weights = np.prod(scales, axis=0)
    order = np.argsort(weights)[::-1]
    return [P[:, order] for P in profiles], weights[order]


poisson_factors, poisson_res = gcp_fit(T, rank=3, loss="poisson", maxiter=800)
M_poisson = np.einsum(cp_einsum(T.ndim), *poisson_factors)
poisson_profiles, poisson_weights = normalise(poisson_factors)

print("converged / convergió:", poisson_res.success,
      " iterations / iteraciones:", poisson_res.nit)
print("prediction range / rango predicho:",
      round(float(M_poisson.min()), 5), "to / a", round(float(M_poisson.max()), 2))
print("weights / pesos:", np.round(poisson_weights, 1))

### Interactive loss explorer / Explorador interactivo de pérdidas

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

One observed count, every possible prediction. The loss curve is what the optimiser is walking down, and the point to look for is where each curve bottoms out and how steeply it climbs on the low side.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Un conteo observado, todas las predicciones posibles. La curva de pérdida es por la que baja el optimizador; lo que hay que mirar es dónde toca fondo cada curva y con cuánta pendiente sube por el lado bajo.</div>

In [ ]:
#@title 📉 Loss explorer / Explorador de pérdidas { display-mode: 'form' }

import ipywidgets as widgets
from IPython.display import display


def loss_draw(observed, loss_name, log_scale):
    value_of, _ = LOSSES[loss_name]
    grid = np.linspace(0.02, max(4.0, 2.5 * observed), 400)
    curve = value_of(float(observed), grid)
    curve = curve - curve.min()

    figure, axes = plt.subplots(figsize=(7.2, 3.4))
    axes.plot(grid, curve, lw=2.2, color="#b91c1c")
    axes.axvline(observed, color="#6b7280", ls="--", lw=1)
    axes.annotate(f"observed x = {observed}", (observed, curve.max() * .85),
                  fontsize=9, color="#6b7280",
                  xytext=(6, 0), textcoords="offset points")
    if log_scale:
        axes.set_yscale("symlog", linthresh=1e-3)
    axes.set_xlabel("prediction m / predicción m")
    axes.set_ylabel("loss above its minimum / pérdida sobre su mínimo")
    axes.set_title(f"{loss_name} — minimised at m = "
                   f"{grid[int(np.argmin(curve))]:.2f}", fontsize=10)
    axes.grid(alpha=.25)
    figure.tight_layout()
    plt.show()


loss_observed = widgets.IntSlider(min=0, max=40, value=2,
                                  description="observed x / observado x:",
                                  style={"description_width": "210px"},
                                  continuous_update=False)
loss_which = widgets.Dropdown(options=list(LOSSES), value="poisson",
                              description="loss / pérdida:",
                              style={"description_width": "210px"})
loss_log = widgets.Checkbox(value=False, indent=False,
                            description="log scale / escala logarítmica")

display(widgets.VBox([
    loss_observed, loss_which, loss_log,
    widgets.interactive_output(
        loss_draw, {"observed": loss_observed, "loss_name": loss_which,
                    "log_scale": loss_log}),
]))

<details>
<summary><strong>What did Exercise 2 show? / ¿Qué mostró el Ejercicio 2?</strong></summary>

**Swapping the loss is two lines.** The model, the unfolding, the Khatri–Rao product, the packing into one vector — none of it knows or cares which `f` it is summing. That is the whole design of GCP, and it is why the paper's Table 1 can be a dozen rows long without a dozen algorithms behind it.

**Every prediction is now positive, and nothing enforced that.** The Poisson loss contains `log m`, which is undefined at zero and enormous just above it, so the optimiser cannot go there. The bound at `EPS` is a numerical guard, not the mechanism — the mechanism is that predicting nearly zero where something was observed is charged an unbounded amount.

**Check the loss explorer at `x = 0`.** The Poisson curve has no barrier on the low side there, because `x log m` vanishes: a cell that was genuinely empty is allowed to be predicted empty, for free. That asymmetry is the whole reason a Poisson fit handles a 41%-zero tensor gracefully and a Gaussian one does not.

**It is slower, and that is the trade.** ALS needed forty sweeps of closed-form solves. This needs several hundred L-BFGS-B iterations, because there is no closed form to jump to. On this tensor that is a second or two; on a tensor a hundred times larger it is the reason stochastic GCP exists.

<br>

**ES.** **Cambiar la pérdida son dos líneas**: ni el modelo, ni el desplegado, ni el Khatri–Rao, ni el empaquetado saben qué `f` están sumando. Ese es todo el diseño de GCP, y por eso la tabla 1 del artículo puede tener doce filas sin doce algoritmos detrás. **Ahora toda predicción es positiva, y nada lo impuso**: la pérdida de Poisson contiene `log m`, indefinido en cero y enorme justo encima, así que el optimizador no puede ir allí; la cota en `EPS` es una guarda numérica, no el mecanismo. **Mira el explorador en `x = 0`**: ahí la curva de Poisson no tiene barrera por abajo, porque `x log m` se anula — una celda genuinamente vacía puede predecirse vacía, gratis. Esa asimetría es la razón de que un ajuste de Poisson maneje bien un tensor con 41% de ceros. **Es más lento, y ese es el intercambio**: no hay forma cerrada a la que saltar.

</details>

## Exercise 3 — read the four profiles / Ejercicio 3 — lee los cuatro perfiles

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

A component of this fit is four vectors: a weekday profile, an hourly profile, a neighbourhood profile and a crime-type profile. Deep dive 14's rule applies unchanged — normalise first, then read the peaks.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una componente de este ajuste son cuatro vectores: un perfil por día de la semana, uno por hora, uno por barrio y uno por tipo de delito. La regla del estudio 14 vale igual: normaliza primero, después lee los picos.</div>

In [ ]:
# TODO 3 / TAREA 3
#
# poisson_profiles holds the four normalised factor matrices, ordered so that
# component 0 is the heaviest. poisson_weights holds the sizes.
#
# For each component r:
#   1. peak_hour[r]  = the hour where its hourly profile peaks.
#   2. peak_day[r]   = DAYS[...] for the day-of-week where its profile peaks.
#   3. top_type[r]   = TYPES[...] for the crime type it loads on most.
#   4. top_areas[r]  = the three community areas it loads on most.
#
# Then print one line per component and write a sentence describing each.
# Which of the three would you be willing to put in front of a city official?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }

day_p, hour_p, area_p, type_p = poisson_profiles

peak_hour = [int(np.argmax(hour_p[:, r])) for r in range(3)]
peak_day = [DAYS[int(np.argmax(day_p[:, r]))] for r in range(3)]
top_type = [TYPES[int(np.argmax(type_p[:, r]))] for r in range(3)]
top_areas = [[AREAS[i] for i in np.argsort(area_p[:, r])[::-1][:3]]
             for r in range(3)]

for r in range(3):
    print(f"component {r}  weight {poisson_weights[r]:8.1f}")
    print(f"   peaks at {peak_hour[r]:02d}:00 on {peak_day[r]}, "
          f"mostly {top_type[r]}")
    print(f"   heaviest community areas / áreas: {top_areas[r]}")
    print("   hourly profile / perfil horario:",
          np.array2string(np.round(hour_p[:, r] / hour_p[:, r].sum(), 3),
                          max_line_width=78))
    print()

In [ ]:
# Visible on purpose: Exercise 4 reads these.
# Visible a propósito: el Ejercicio 4 los lee.

day_p, hour_p, area_p, type_p = poisson_profiles

peak_hour = [int(np.argmax(hour_p[:, r])) for r in range(3)]
peak_day = [DAYS[int(np.argmax(day_p[:, r]))] for r in range(3)]
top_type = [TYPES[int(np.argmax(type_p[:, r]))] for r in range(3)]

for r in range(3):
    print(f"component {r}: {peak_hour[r]:02d}:00 · {peak_day[r]} · "
          f"{top_type[r]}   weight {poisson_weights[r]:.1f}")

### Interactive profile explorer / Explorador interactivo de perfiles

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

The four profiles of one component, and the real marginal counts underneath for comparison. If a component's hourly profile looks nothing like the city's actual hourly pattern, that is the interesting case.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Los cuatro perfiles de una componente, con los conteos marginales reales debajo para comparar. Si el perfil horario de una componente no se parece en nada al patrón horario real de la ciudad, ese es el caso interesante.</div>

In [ ]:
#@title 🕐 Profile explorer / Explorador de perfiles { display-mode: 'form' }

import ipywidgets as widgets
from IPython.display import display

hour_marginal = T.sum(axis=(0, 2, 3))
day_marginal = T.sum(axis=(1, 2, 3))


def profile_draw(component, show_marginal):
    r = component
    figure, axes = plt.subplots(1, 3, figsize=(11.6, 2.9))

    axes[0].bar(np.arange(7), day_p[:, r] / day_p[:, r].sum(), color="#b91c1c")
    if show_marginal:
        axes[0].plot(np.arange(7), day_marginal / day_marginal.sum(),
                     color="#6b7280", lw=1.6, marker="o", ms=3)
    axes[0].set_xticks(np.arange(7))
    axes[0].set_xticklabels(DAYS, fontsize=7)
    axes[0].set_title("day of week / día", fontsize=10)

    axes[1].bar(np.arange(24), hour_p[:, r] / hour_p[:, r].sum(),
                color="#b91c1c")
    if show_marginal:
        axes[1].plot(np.arange(24), hour_marginal / hour_marginal.sum(),
                     color="#6b7280", lw=1.6)
    axes[1].set_xticks([0, 6, 12, 18, 23])
    axes[1].set_title("hour / hora", fontsize=10)

    share = type_p[:, r] / type_p[:, r].sum()
    top = np.argsort(share)[::-1][:5]
    axes[2].barh(np.arange(len(top)), share[top][::-1], color="#b91c1c")
    axes[2].set_yticks(np.arange(len(top)))
    axes[2].set_yticklabels([TYPES[i].title() for i in top][::-1], fontsize=7)
    axes[2].set_title("crime type / tipo", fontsize=10)

    for ax in axes[:2]:
        ax.set_yticks([])
        ax.grid(alpha=.2)
    figure.suptitle(f"component {r} · weight {poisson_weights[r]:.1f} · "
                    f"grey line = the city's own marginal", fontsize=10)
    figure.tight_layout()
    plt.show()


profile_pick = widgets.IntSlider(min=0, max=2, value=0,
                                 description="component / componente:",
                                 style={"description_width": "200px"},
                                 continuous_update=False)
profile_marginal = widgets.Checkbox(value=True, indent=False,
                                    description="show the city's marginal / "
                                                "muestra el marginal real")

display(widgets.VBox([
    profile_pick, profile_marginal,
    widgets.interactive_output(
        profile_draw, {"component": profile_pick,
                       "show_marginal": profile_marginal}),
]))

<details>
<summary><strong>What did Exercise 3 show? / ¿Qué mostró el Ejercicio 3?</strong></summary>

Three components, and two of them read like a city. One peaks in the late evening and loads on theft; one peaks around midnight and loads on battery, with a weekday profile that leans towards the weekend. Both have hourly profiles that rise and fall smoothly, and both have a clear trough in the small hours — the shape you would draw from memory if somebody asked you when crime happens.

The third does not look like that at all, and Exercise 4 is about it.

A note on what the weights mean here. Under a Poisson loss the model value is a **rate**, so a component's weight is roughly how many reports it accounts for across the year. That makes the weights comparable in a way the Gaussian fit's were not, and it is a small, real benefit of choosing the loss that matches the data: the number on the front of a component has units.

<br>

**ES.** Tres componentes, y dos se leen como una ciudad: una culmina al final de la tarde y carga sobre el hurto; otra culmina cerca de medianoche y carga sobre las agresiones, con un perfil semanal escorado al fin de semana. Las dos tienen perfiles horarios que suben y bajan con suavidad y un valle claro de madrugada: la forma que dibujarías de memoria. La tercera no se parece en nada a eso, y de ella trata el Ejercicio 4. Sobre los pesos: bajo una pérdida de Poisson el valor del modelo es una **tasa**, así que el peso de una componente es aproximadamente cuántas denuncias explica en el año. Eso hace los pesos comparables como no lo eran los del ajuste gaussiano, y es un beneficio pequeño y real de elegir la pérdida que corresponde a los datos: el número que encabeza una componente tiene unidades.

</details>

## Exercise 4 — a component that is not about crime / Ejercicio 4 — una componente que no trata de delitos

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

One of the three has an hourly profile that does something no human schedule does: it spikes at exactly one hour, collapses to almost nothing for the next five, climbs back through the morning, and then spikes again at exactly one other hour — twice as high as the hours either side of it. Find it, and work out what it is.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una de las tres tiene un perfil horario que hace algo que ningún horario humano hace: se dispara en una hora exacta, se derrumba casi a cero durante las cinco siguientes, remonta a lo largo de la mañana y vuelve a dispararse en otra hora exacta, al doble que las horas de al lado. Encuéntrala y averigua qué es.</div>

In [ ]:
# TODO 4 / TAREA 4
#
# 1. spike_score[r] = a number that is large when an hourly profile is
#    concentrated on a few hours and small when it is smooth. One that works:
#       the profile's largest value divided by its median.
#
# 2. odd = the component with the largest spike_score.
#
# 3. Print odd's hourly profile as 24 numbers. Which two hours carry it?
#    What is the profile's value in the surrounding hours?
#
# 4. Look up the crime type it loads on most. Then answer, in a sentence:
#    is this a pattern in the city, or a pattern in the file? What would you
#    have to do to the data before this component stopped appearing?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }

shares = hour_p / hour_p.sum(axis=0)
spike_score = shares.max(axis=0) / np.median(shares, axis=0)
odd = int(np.argmax(spike_score))

print("spike score per component / puntuación de pico:",
      np.round(spike_score, 1))
print("the odd one out / la rara:", odd)
print()
print("hourly profile / perfil horario:")
for hour in range(24):
    bar = "█" * int(round(120 * shares[hour, odd]))
    print(f"  {hour:02d}:00  {shares[hour, odd]:.3f}  {bar}")
print()
type_share = type_p[:, odd] / type_p[:, odd].sum()
print("crime types, largest share first / tipos de delito, por cuota:")
for i in np.argsort(type_share)[::-1][:4]:
    print(f"   {TYPES[i]:<20} {type_share[i]:.3f}")
print("midnight and noon together / medianoche y mediodía juntos:",
      round(float(shares[0, odd] + shares[12, odd]), 3))
print("hours 02:00-05:00 together / horas 02:00-05:00 juntas:",
      round(float(shares[2:6, odd].sum()), 3))

In [ ]:
# Visible on purpose: the wrap-up reads these.
# Visible a propósito: el cierre los lee.

shares = hour_p / hour_p.sum(axis=0)
spike_score = shares.max(axis=0) / np.median(shares, axis=0)
odd = int(np.argmax(spike_score))

print("the odd component / la componente rara:", odd,
      " spike score / puntuación:", round(float(spike_score[odd]), 1))
print("share at 00:00 / cuota a las 00:00:", round(float(shares[0, odd]), 3))
print("share at 12:00 / cuota a las 12:00:", round(float(shares[12, odd]), 3))
print("share at 05:00 / cuota a las 05:00:", round(float(shares[5, odd]), 3))

<details>
<summary><strong>What did Exercise 4 show? / ¿Qué mostró el Ejercicio 4?</strong></summary>

Its two tallest hours are **00:00 and 12:00**, together carrying a fifth of the component, while 05:00 carries about half a percent. Nothing in a city does that. Crime has a smooth daily rhythm — the other two components show it, and so does the grey marginal line in the profile explorer.

What does do that is a **form**. When the time of an offence is not known, it gets recorded as midnight or noon, because those are what a blank time field defaults to and what a person writing "sometime that day" puts down.

Read the crime-type profile the solution prints and the story holds up. Theft leads it, which is not surprising — theft is the commonest category in the tensor, so it leads most things. What is telling is the *second* entry: deceptive practice takes about a quarter of the component, far more than its share of the corpus, and it is exactly the category where the victim discovers the offence later and cannot say when it happened. Both belong to the same mechanism: this component is not sorted by what the crime was, it is sorted by **whether anybody knew what time it happened**.

So the decomposition found a real, strong, reproducible pattern, and the pattern is in the recording process rather than in the city. That is not a failure. **It is the most useful thing a rank-3 summary could have told you about this dataset**, and it is invisible in the raw table: no single row looks wrong, and the midnight spike is only 3% of all reports. It takes a method that looks at day, hour, place and type together to make it stand out as its own component.

The lesson generalises past this dataset, and it is section 10's rule met on new data: **a component is a pattern in the numbers you were given.** Whether it is a pattern in the world is a separate question, and it is yours, not the method's. Before quoting a component, ask what would have to be true of the *collection* for it to look like this — and check the flattest, most obviously-artificial answer first.

To get rid of it you would have to decide what those reports mean. Dropping every offence timestamped exactly 00:00 or 12:00 throws away real crimes that genuinely happened then. Treating those cells as **missing** is the honest option, and generalized CP handles that directly — the note below the next exercise says how.

<br>

**ES.** Sus dos horas más altas son las **00:00 y las 12:00**, que juntas llevan una quinta parte de la componente, mientras las 05:00 llevan medio por ciento. Nada en una ciudad hace eso; un **formulario** sí. Cuando no se conoce la hora de un delito se registra a medianoche o mediodía, porque es lo que pone un campo vacío y lo que escribe quien declara «en algún momento de ese día». Lee el perfil de tipos que imprime la solución: encabeza el hurto, lo que no sorprende porque es la categoría más común del tensor. Lo revelador es la *segunda* entrada: la práctica engañosa se lleva alrededor de un cuarto de la componente, mucho más que su cuota del corpus, y es justo la categoría en que la víctima descubre el delito después y no puede decir cuándo ocurrió. Ambas pertenecen al mismo mecanismo: esta componente no está ordenada por qué fue el delito, sino por <b>si alguien supo a qué hora ocurrió</b>.
<br><br>
Así que la descomposición encontró un patrón real, fuerte y reproducible, y el patrón está en el proceso de registro, no en la ciudad. No es un fallo: **es lo más útil que un resumen de rango 3 podía decirte de estos datos**, y es invisible en la tabla cruda. La lección generaliza, y es la regla de la sección 10 sobre datos nuevos: **una componente es un patrón en los números que te dieron**; si además es un patrón del mundo es otra pregunta, y es tuya. Para quitarla habría que decidir qué significan esas denuncias: tirar todo lo marcado a las 00:00 exactas descarta delitos que sí ocurrieron entonces. Tratar esas celdas como **ausentes** es la opción honesta, y el CP generalizado lo admite directamente.

</details>

## Exercise 5 — the same tensor, made binary / Ejercicio 5 — el mismo tensor, hecho binario

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

Sometimes the count is not the question. *Did this kind of crime ever happen in this neighbourhood, at this hour, on this day of the week?* is a different question, and its answer is a tensor of ones and zeros — for which neither a Gaussian nor a Poisson model is right.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>A veces el conteo no es la pregunta. <i>¿Ocurrió alguna vez este tipo de delito en este barrio, a esta hora, este día de la semana?</i> es otra pregunta, y su respuesta es un tensor de unos y ceros, para el que no vale ni un modelo gaussiano ni uno de Poisson.</div>

In [ ]:
# TODO 5 / TAREA 5
#
# 1. B = (T > 0), as floats. What fraction of it is ones?
#
# 2. Fit it with the squared-error loss and positive=False. How many of the
#    131,040 predictions fall outside [0, 1]? What are the smallest and
#    largest?
#
# 3. Fit it with the Bernoulli loss. The model value m is the *odds*, so the
#    probability is m / (1 + m). Confirm every probability is inside (0, 1),
#    and say why that needed no constraint.
#
# 4. Read the Bernoulli fit's hourly profiles. How do they differ from the
#    Poisson fit's, and why would they? Bernoulli maxiter=3000 takes about
#    fifteen seconds -- the surface is flatter than Poisson's near the optimum.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }

B = (T > 0).astype(float)
print("ones / unos:", f"{100 * B.mean():.1f}%")

sq_factors, _ = gcp_fit(B, rank=3, loss="gaussian", maxiter=800,
                        positive=False)
M_sq = np.einsum(cp_einsum(B.ndim), *sq_factors)
outside = int(((M_sq < 0) | (M_sq > 1)).sum())
print()
print("squared error / error cuadrático")
print("   range / rango:", round(float(M_sq.min()), 3), "to / a",
      round(float(M_sq.max()), 3))
print("   outside [0, 1] / fuera de [0, 1]:", outside, "of / de", M_sq.size)

bern_factors, bern_res = gcp_fit(B, rank=3, loss="bernoulli", maxiter=3000)
M_bern = np.einsum(cp_einsum(B.ndim), *bern_factors)
P_bern = M_bern / (1 + M_bern)
print()
print("Bernoulli / Bernoulli   converged:", bern_res.success,
      " iterations:", bern_res.nit)
print("   probability range / rango de probabilidad:",
      round(float(P_bern.min()), 3), "to / a", round(float(P_bern.max()), 3))
print("   outside [0, 1] / fuera de [0, 1]:",
      int(((P_bern < 0) | (P_bern > 1)).sum()))

bern_profiles, bern_weights = normalise(bern_factors)
print()
print("Bernoulli components / componentes de Bernoulli")
for r in range(3):
    hours = bern_profiles[1][:, r]
    print(f"   {r}: peaks at {int(np.argmax(hours)):02d}:00, "
          f"mostly {TYPES[int(np.argmax(bern_profiles[3][:, r]))]}")

<details>
<summary><strong>What did Exercise 5 show? / ¿Qué mostró el Ejercicio 5?</strong></summary>

**Squared error on a 0/1 tensor predicts values it has no meaning for.** About a tenth of the predictions fall outside `[0, 1]` — below zero and above one — and there is no reading of "probability" or "did it happen" that accommodates 1.24. The fit is not broken; it was asked the wrong question and answered it correctly.

**The Bernoulli fit cannot leave the interval, and nothing bounds it.** The model value `m` is the *odds*, which the loss keeps positive, and `m / (1 + m)` maps any positive number into `(0, 1)`. The constraint is in the parameterisation rather than in the optimiser — which is the same trick logistic regression uses, arriving here by the same route.

**Binarising changes what the tensor is about, so of course the components change.** Counting weights a busy neighbourhood-hour heavily; asking only *whether* something happened gives a quiet hour and a busy one the same vote. The Bernoulli components lean towards what is *widespread* rather than what is *frequent*, and both fits are correct about different questions. Choosing between them is a modelling decision and belongs in your write-up, not in a footnote.

<br>

**ES.** **El error cuadrático sobre un tensor de 0 y 1 predice valores que no significan nada**: cerca de una décima parte de las predicciones cae fuera de `[0, 1]`, y no hay lectura de «probabilidad» que admita 1,24. El ajuste no está roto: se le hizo la pregunta equivocada y la respondió bien. **El ajuste de Bernoulli no puede salirse del intervalo, y nada lo acota**: el valor del modelo `m` son las probabilidades relativas, que la pérdida mantiene positivas, y `m / (1 + m)` lleva cualquier positivo a `(0, 1)`. La restricción está en la parametrización, no en el optimizador — el mismo truco de la regresión logística. **Binarizar cambia de qué trata el tensor**, así que las componentes cambian: contar pesa mucho una hora concurrida; preguntar solo *si* ocurrió da el mismo voto a una hora tranquila. Bernoulli se inclina hacia lo *extendido* y Poisson hacia lo *frecuente*, y los dos aciertan en preguntas distintas.

</details>

## Exercise 6 — the same objective, handed to autograd / Ejercicio 6 — el mismo objetivo, entregado a autograd

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

`gcp_fit` works because you wrote `∂f/∂m` by hand and turned it into a factor gradient with one Khatri–Rao product. That is worth having done once. It is also the part you do not want to redo every time you try a new loss, and it is the part that stops scaling first.

So: write the **objective only**, in PyTorch, and let autograd produce the gradient. Nothing about the model changes. What you get in exchange is the whole optimiser shelf — plain gradient descent, momentum, Adam — and a straight route to a GPU and to minibatches over a tensor too large to hold at once, which is what Hong, Kolda and Duersch's stochastic variant is for.

One change to the *coordinates*, and it is not cosmetic. Store each factor as `exp(θ)` rather than as itself. The model is identical — `exp` of a real number is a positive number, which is all the Poisson loss asked for — but there is no longer a constraint for the optimiser to bump into, and the gradient stays well-scaled instead of exploding as a factor entry approaches zero. Reparameterising does not change the model; it changes what gradient descent can see.

`torch` ships with Colab. Locally the install below is a large download; skip this exercise if you would rather not.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>gcp_fit</code> funciona porque escribiste <code>∂f/∂m</code> a mano y lo convertiste en un gradiente de factores con un producto Khatri–Rao. Merece la pena haberlo hecho una vez. También es la parte que no quieres rehacer con cada pérdida nueva, y la primera que deja de escalar.
<br><br>
Escribe entonces <b>solo el objetivo</b>, en PyTorch, y deja que autograd produzca el gradiente. El modelo no cambia en nada. A cambio obtienes toda la estantería de optimizadores — descenso simple, momento, Adam — y un camino directo a la GPU y a minilotes sobre un tensor demasiado grande para caber entero, que es para lo que existe la variante estocástica de Hong, Kolda y Duersch.
<br><br>
Un cambio en las <i>coordenadas</i>, y no es cosmético: guarda cada factor como <code>exp(θ)</code> en vez de como sí mismo. El modelo es idéntico — el <code>exp</code> de un número real es un número positivo, que es todo lo que pedía la pérdida de Poisson — pero ya no hay restricción con la que tropezar, y el gradiente se mantiene bien escalado en vez de dispararse cuando una entrada se acerca a cero. Reparametrizar no cambia el modelo: cambia lo que el descenso por gradiente puede ver.
<br><br>
<code>torch</code> viene con Colab. En local la instalación de abajo es una descarga grande; sáltate este ejercicio si prefieres no hacerla.</div>

In [ ]:
# Colab already has torch; this is a no-op there and a large download locally.
# Colab ya trae torch; aquí no hace nada y en local es una descarga grande.
%pip install -q torch

import torch

print("torch / torch:", torch.__version__)

In [ ]:
# TODO 6 / TAREA 6
#
# 1. torch_gcp(A, rank, loss, steps, lr, optimiser, seed) -> (factors, curve)
#
#    Hold one torch.nn.Parameter per axis, of shape (dim, rank), and treat it
#    as log-factors: the factor itself is theta.exp(), so it is positive and
#    unconstrained. Start every entry near log(mean / rank) / ndim.
#
#    Each step: rebuild the model with torch.einsum(cp_einsum(A.ndim), ...),
#    evaluate the loss as one expression, call .backward(), and step. Record
#    the loss. No gradient is written by you.
#
#    Support three optimisers: "sgd", "momentum" (SGD with momentum=0.9) and
#    "adam".
#
# 2. Fit rank 3 with the Poisson loss and Adam at lr=0.05 for 1500 steps.
#    How close does it get to gcp_fit's L-BFGS-B objective?
#
# 3. Now try "sgd" at the same lr. Then find a learning rate at which it does
#    not diverge. How much smaller is it, and where does it finish?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }


def torch_loss(name, X, M):
    """The same three losses, as one torch expression each."""
    if name == "gaussian":
        return ((X - M) ** 2).sum()
    if name == "poisson":
        return (M - X * torch.log(M + EPS)).sum()
    return (torch.log(M + 1) - X * torch.log(M + EPS)).sum()


def torch_gcp(A, rank, loss="poisson", steps=1500, lr=0.05,
              optimiser="adam", seed=15):
    """GCP by autograd. Factors are exp(theta): positive, and unconstrained."""
    X = torch.tensor(A, dtype=torch.float64)
    generator = torch.Generator().manual_seed(seed)
    start = np.log(max(A.mean(), EPS) / rank) / A.ndim
    theta = [torch.nn.Parameter(
        start + 0.1 * torch.randn(dim, rank, generator=generator,
                                  dtype=torch.float64))
        for dim in A.shape]

    step = {
        "sgd": lambda: torch.optim.SGD(theta, lr=lr),
        "momentum": lambda: torch.optim.SGD(theta, lr=lr, momentum=0.9),
        "adam": lambda: torch.optim.Adam(theta, lr=lr),
    }[optimiser]()

    spec = cp_einsum(A.ndim)
    curve = []
    for _ in range(steps):
        step.zero_grad()
        value = torch_loss(loss, X, torch.einsum(spec, *[t.exp() for t in theta]))
        value.backward()          # the gradient you wrote by hand, for free
        step.step()
        curve.append(value.item())

    return [t.detach().exp().numpy() for t in theta], np.array(curve)


adam_factors, adam_curve = torch_gcp(T, 3, "poisson", steps=1500, lr=0.05,
                                     optimiser="adam")
print("Adam, lr 0.05      :", round(float(adam_curve[-1]), 1))
print("scipy L-BFGS-B     :", round(float(poisson_res.fun), 1))
print()
for lr in (0.05, 1e-4, 1e-5):
    _, curve = torch_gcp(T, 3, "poisson", steps=1500, lr=lr, optimiser="sgd")
    print(f"plain SGD, lr {lr:<7}:", round(float(curve[-1]), 1))

In [ ]:
# Visible on purpose: the explorer below needs these.
# Visible a propósito: el explorador de abajo los necesita.


def torch_loss(name, X, M):
    """The same three losses, as one torch expression each."""
    if name == "gaussian":
        return ((X - M) ** 2).sum()
    if name == "poisson":
        return (M - X * torch.log(M + EPS)).sum()
    return (torch.log(M + 1) - X * torch.log(M + EPS)).sum()


def torch_gcp(A, rank, loss="poisson", steps=1500, lr=0.05,
              optimiser="adam", seed=15):
    """GCP by autograd. Factors are exp(theta): positive, and unconstrained."""
    X = torch.tensor(A, dtype=torch.float64)
    generator = torch.Generator().manual_seed(seed)
    start = np.log(max(A.mean(), EPS) / rank) / A.ndim
    theta = [torch.nn.Parameter(
        start + 0.1 * torch.randn(dim, rank, generator=generator,
                                  dtype=torch.float64))
        for dim in A.shape]
    step = {
        "sgd": lambda: torch.optim.SGD(theta, lr=lr),
        "momentum": lambda: torch.optim.SGD(theta, lr=lr, momentum=0.9),
        "adam": lambda: torch.optim.Adam(theta, lr=lr),
    }[optimiser]()
    spec = cp_einsum(A.ndim)
    curve = []
    for _ in range(steps):
        step.zero_grad()
        value = torch_loss(loss, X, torch.einsum(spec, *[t.exp() for t in theta]))
        value.backward()
        step.step()
        curve.append(value.item())
    return [t.detach().exp().numpy() for t in theta], np.array(curve)


torch_runs = {}
for name, rate in (("adam", 0.05), ("momentum", 1e-5), ("sgd", 1e-5),
                   ("momentum", 1e-4), ("sgd", 1e-4)):
    torch_runs[(name, rate)] = torch_gcp(T, 3, "poisson", steps=1500, lr=rate,
                                         optimiser=name)[1]

print("objective after 1500 steps / objetivo tras 1500 pasos")
for (name, rate), curve in torch_runs.items():
    print(f"   {name:<9} lr {rate:<8} {curve[-1]:14.1f}")
print(f"   {'L-BFGS-B':<9} {'':<11} {poisson_res.fun:14.1f}")

### Interactive optimiser explorer / Explorador interactivo de optimizadores

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

The same objective, the same starting point, three optimisers. The dashed line is what L-BFGS-B reached. Watch what plain gradient descent needs before it stops diverging, and what momentum buys at the same step size.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El mismo objetivo, el mismo punto de partida, tres optimizadores. La línea discontinua es lo que alcanzó L-BFGS-B. Fíjate en lo que necesita el descenso simple para dejar de divergir, y en lo que compra el momento con el mismo tamaño de paso.</div>

In [ ]:
#@title 🏃 Optimiser explorer / Explorador de optimizadores { display-mode: 'form' }

import ipywidgets as widgets
from IPython.display import display

# Precomputed above: moving a control is a lookup, not a fit.
# Precalculado arriba: mover un control es una consulta, no un ajuste.


def optimiser_draw(rate, show_all):
    figure, axes = plt.subplots(figsize=(7.4, 3.6))
    shown = [("adam", 0.05)] if not show_all else list(torch_runs)
    if (("sgd", rate) in torch_runs) and not show_all:
        shown += [("sgd", rate), ("momentum", rate)]
    colours = {"adam": "#b91c1c", "momentum": "#2563eb", "sgd": "#6b7280"}
    for key in shown:
        curve = torch_runs[key]
        clipped = np.clip(curve, poisson_res.fun - 100, None)
        axes.plot(clipped, lw=1.8, color=colours[key[0]],
                  alpha=.95 if key[0] == "adam" else .7,
                  label=f"{key[0]}, lr {key[1]:g}")
    axes.axhline(poisson_res.fun, color="#111827", ls="--", lw=1,
                 label="L-BFGS-B")
    axes.set_yscale("symlog", linthresh=1e4)
    axes.set_xlabel("step / paso")
    axes.set_ylabel("objective / objetivo")
    axes.legend(fontsize=8, loc="upper right")
    axes.grid(alpha=.25)
    figure.tight_layout()
    plt.show()


opt_rate = widgets.SelectionSlider(options=[1e-5, 1e-4], value=1e-5,
                                   description="SGD lr / paso SGD:",
                                   style={"description_width": "190px"})
opt_all = widgets.Checkbox(value=False, indent=False,
                           description="show every run / muestra todas")

display(widgets.VBox([
    opt_rate, opt_all,
    widgets.interactive_output(
        optimiser_draw, {"rate": opt_rate, "show_all": opt_all}),
]))

<details>
<summary><strong>What did Exercise 6 show? / ¿Qué mostró el Ejercicio 6?</strong></summary>

**Adam lands where L-BFGS-B did, and you wrote no gradient.** Two expressions — the model and the loss — and `backward()` produced everything `gcp_fit` spent twenty lines assembling. The Khatri–Rao product is still happening; autograd derived it from the `einsum`.

**Plain gradient descent needs a learning rate about five thousand times smaller, and still finishes short.** At Adam's `lr = 0.05` it diverges immediately; at `1e-4` it still diverges; at `1e-5` it converges and stops above the L-BFGS-B objective. The reason is that this problem's gradient has wildly different scales across the four factor matrices — 7 days against 78 community areas — and a single global step size has to be small enough for the steepest of them.

**Momentum recovers most of the gap for free.** At the same `1e-5` where plain SGD stalls, SGD with `momentum=0.9` gets close to the L-BFGS-B value. Momentum accumulates a running average of past gradients, so consistent directions build up speed while directions that keep reversing cancel out — which is exactly the shape of a badly scaled problem. Adam goes further and keeps a *per-parameter* step size, which is why it tolerates a learning rate two orders of magnitude larger than anything SGD survives.

**This is the door to the big-tensor case.** Once the objective is a differentiable expression, you can evaluate it on a random sample of entries instead of all of them and take a step from that — stochastic GCP, which is what you need when the tensor does not fit in memory. It is also, incidentally, why deep-learning frameworks are a reasonable place to fit a tensor decomposition at all: the optimiser, the sampling and the GPU are already there.

<br>

**ES.** **Adam llega donde llegó L-BFGS-B, y no escribiste ningún gradiente**: dos expresiones — el modelo y la pérdida — y `backward()` produjo todo lo que `gcp_fit` montaba en veinte líneas. El producto Khatri–Rao sigue ocurriendo; autograd lo dedujo del `einsum`. **El descenso simple necesita un paso unas cinco mil veces menor y aun así se queda corto**: con el `lr = 0,05` de Adam diverge de inmediato, con `1e-4` también, y con `1e-5` converge por encima del objetivo de L-BFGS-B. La razón es que el gradiente tiene escalas muy distintas entre las cuatro matrices factor — 7 días frente a 78 áreas — y un único paso global ha de ser lo bastante pequeño para la más empinada. **El momento recupera casi toda la diferencia gratis**: con el mismo `1e-5`, SGD con `momentum=0.9` se acerca al valor de L-BFGS-B, porque acumula una media de gradientes pasados y las direcciones consistentes ganan velocidad mientras las que se invierten se cancelan. Adam va más allá y mantiene un paso *por parámetro*, y por eso tolera un paso dos órdenes de magnitud mayor. **Esta es la puerta al caso de tensores grandes**: con el objetivo como expresión derivable, puedes evaluarlo sobre una muestra aleatoria de entradas — el GCP estocástico, que es lo que hace falta cuando el tensor no cabe en memoria.

</details>

### Missing data is one more term you do not add / Los datos ausentes son un término que no sumas

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

Exercise 4 ended on an honest problem: the reports timestamped exactly midnight and noon are not counts of anything, but deleting them deletes real crimes too. The right description is that **the hour is missing** for those reports, and generalized CP handles that without any new machinery.

The objective is a sum over entries. Drop the entries you do not trust:

$$
F(\mathcal{M}) \;=\; \sum_{i \,\in\, \Omega} f\bigl(x_i,\; m_i\bigr)
$$

where $\Omega$ is the set of observed entries. In code that is one elementwise multiplication by a 0/1 mask, in both the value and the gradient — `gcp_fit` becomes a masked fit in two lines. The model still predicts a value for every cell, including the masked ones, which is how the same change turns a decomposition into a **recommender**: leave out what you have not seen, fit the rest, read off what the model expects.

In PyTorch it is the same one multiplication, and the `mask` goes inside the expression `backward()` differentiates — nothing else changes.

`pyttb`, Kolda's own Tensor Toolbox for Python, takes it as a `mask=` argument to `gcp_opt`, alongside the dozen losses from the paper's Table 1 and a stochastic optimiser for tensors far larger than this one. Between them those are the two libraries to reach for on real work — `pyttb` when you want the paper's algorithms as written, PyTorch when you want a GPU, a custom loss or a minibatch. The seventy lines above are here so you know what both of them are doing.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El Ejercicio 4 terminó en un problema honesto: las denuncias marcadas exactamente a medianoche y mediodía no cuentan nada, pero borrarlas borra también delitos reales. La descripción correcta es que <b>la hora está ausente</b> en esas denuncias, y el CP generalizado lo admite sin maquinaria nueva.
<br><br>
El objetivo es una suma sobre entradas: quita las que no te fías. En código es una multiplicación elemento a elemento por una máscara de 0 y 1, en el valor y en el gradiente — <code>gcp_fit</code> se vuelve un ajuste enmascarado en dos líneas. El modelo sigue prediciendo un valor en todas las celdas, incluidas las enmascaradas, que es como el mismo cambio convierte una descomposición en un <b>recomendador</b>.
<br><br>
<code>pyttb</code>, el Tensor Toolbox de la propia Kolda para Python, lo toma como argumento <code>mask=</code> de <code>gcp_opt</code>, junto a la docena de pérdidas de la tabla 1 del artículo y un optimizador estocástico para tensores mucho mayores que este. Esa es la librería para trabajo real; las treinta líneas de arriba están aquí para que sepas qué hace.</div>

In [ ]:
#@title 🔬 Optional: the same fit in Kolda's pyttb / Opcional: el mismo ajuste en pyttb { display-mode: 'form' }

# Not part of any exercise, and deliberately outside every core path: this
# cell checks our thirty lines against the reference implementation.
#
# Version note, and it is the reason this cell is optional. pyttb 1.8.5 calls
# np.reshape(newshape=...), which NumPy removed in 2.1, so on a newer NumPy it
# raises TypeError on any tensor large enough to reach that code path -- this
# one is. The fix is in pyttb's repository but not yet in a release, so if the
# cell reports that, nothing is wrong with your notebook. Either move on, or
# install the fixed version:
#     %pip install -q pyttb@git+https://github.com/sandialabs/pyttb
# (no quotes on that line on purpose: gen_tables.py harvests URLs from
#  string literals to find the datasets a notebook downloads, and a
#  quoted URL in a comment is indistinguishable to it from a real one.)
# Nota de versión: pyttb 1.8.5 falla con NumPy >= 2.1; no es un fallo tuyo.

%pip install -q pyttb

import logging

try:
    import pyttb as ttb
    from pyttb.gcp.handles import Objectives
    from pyttb.gcp.optimizers import LBFGSB

    # pyttb calls logging.warning() directly, so its F-ordering notice lands on
    # the root logger once per call. Turn the root logger down for the fit and
    # put it back afterwards: this cell is optional and runs in the middle of a
    # live kernel, and leaving it turned down would swallow every later warning
    # in the notebook.
    root = logging.getLogger()
    was = root.level

    try:
        root.setLevel(logging.ERROR)
        reference, _, _ = ttb.gcp_opt(
            ttb.tensor(np.ascontiguousarray(T)), rank=3,
            objective=Objectives.POISSON, optimizer=LBFGSB(maxiter=800),
            printitn=0)
    finally:
        root.setLevel(was)
    ref_hours = reference.factor_matrices[1]
    ref_hours = ref_hours / ref_hours.sum(axis=0)

    ours = hour_p / hour_p.sum(axis=0)
    match = np.array([[abs(np.corrcoef(ours[:, i], ref_hours[:, j])[0, 1])
                       for j in range(3)] for i in range(3)])
    print("hourly-profile |correlation| against pyttb, best match per component")
    print("   ", np.round(match.max(axis=1), 3))
    print("EN: values near 1 mean the two implementations found the same "
          "components, up to the reordering deep dive 14 warned about.")
    print("ES: valores cercanos a 1 significan que ambas implementaciones "
          "encontraron las mismas componentes, salvo el reordenamiento.")
except Exception as error:
    print("EN: pyttb did not run here —", type(error).__name__, str(error)[:120])
    # The install command is in the comment at the top of this cell rather
    # than in this string: a URL inside a string literal is how gen_tables.py
    # finds the datasets a notebook downloads, and this is not one.
    print("EN: on NumPy 2.1 or newer this is the known version problem above, "
          "not a mistake in the notebook. To run it anyway, install pyttb from "
          "its repository with the command in the comment at the top of this "
          "cell.")
    print("ES: pyttb no se ejecutó aquí. Con NumPy 2.1 o posterior es el "
          "problema de versión descrito arriba, no un error del cuaderno. Para "
          "ejecutarlo igualmente, instala la versión corregida desde el "
          "repositorio de pyttb.")

## What just happened / Qué acaba de pasar

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

You found out what squared error had been assuming on your behalf since section 07, and watched it fail on its own terms: a rank-3 fit of a count tensor predicting negative crimes, on more than a thousand cells, nearly all of them cells that were empty.

You replaced it. The CP model never changed — the same factors, the same rank-1 terms, the same profiles to read. One elementwise function changed, and with it the question the fit was answering. The alternating structure survived because these losses stay convex in one factor with the rest fixed; the closed form did not, so a general optimiser does the work that a pseudoinverse used to.

You fitted counts with a Poisson loss and read four profiles off each component — a day, an hour, a neighbourhood, a crime type — and found that one of the three components is not about crime at all. It is a picture of what a form does when nobody knows the time, and it was strong enough to claim a third of a rank-3 budget.

Then you asked a different question of the same data, got a tensor of ones and zeros, and saw a squared-error fit answer it with 1.24.

Finally you gave the objective away. Written as two lines of PyTorch it needs no gradient from you at all, and every optimiser on the shelf becomes available — which matters, because this problem is badly enough scaled that plain gradient descent needs a step five thousand times smaller than Adam's and still finishes short. Momentum closes most of that gap; a per-parameter step closes the rest. And a differentiable objective is what lets you evaluate on a sample of entries rather than all of them, which is the only way the largest of these tensors get fitted at all.

**The sentence to remember:** *the loss is where you tell the decomposition what kind of number it is looking at.*

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Descubriste qué llevaba suponiendo el error cuadrático en tu nombre desde la sección 07 y lo viste fallar en sus propios términos: un ajuste de rango 3 de un tensor de conteos prediciendo delitos negativos en más de mil celdas, casi todas vacías.
<br><br>
Lo sustituiste. El modelo CP no cambió nunca: los mismos factores, los mismos términos de rango 1, los mismos perfiles que leer. Cambió una función elemento a elemento, y con ella la pregunta que respondía el ajuste. La estructura alterna sobrevivió porque estas pérdidas siguen siendo convexas en un factor con los demás fijos; la forma cerrada no, así que un optimizador general hace el trabajo que hacía una pseudoinversa.
<br><br>
Ajustaste conteos con una pérdida de Poisson y leíste cuatro perfiles por componente — un día, una hora, un barrio, un tipo de delito — y encontraste que una de las tres no trata de delitos en absoluto: es un retrato de lo que hace un formulario cuando nadie sabe la hora, y fue lo bastante fuerte como para llevarse un tercio de un presupuesto de rango 3. Después le hiciste otra pregunta a los mismos datos, obtuviste un tensor de unos y ceros y viste a un ajuste de error cuadrático responderla con 1,24.
<br><br>
<b>La frase que recordar:</b> <i>la pérdida es donde le dices a la descomposición qué clase de número está mirando.</i></div>

Both deep dives lean on the same two papers. Kolda and Bader's 2009 survey is where CP, Tucker and the rest are laid out together; Hong, Kolda and Duersch's 2020 paper is where the loss becomes a choice. Both are on the [references page](https://project-delphi.github.io/tensors-workshop/references.html), with what each one is for.

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#b91c1c,rgba(185,28,28,0))"></div>

## Done with this deep dive / Fin de este estudio a fondo

That is the last deep dive. The rest is back on [the workshop site](https://project-delphi.github.io/tensors-workshop/).

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Ese es el último estudio a fondo. El resto está en <a href="https://project-delphi.github.io/tensors-workshop/">el sitio del taller</a>.</div></div>

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)